# Multimodal Contradiction Project

This notebook implements the project pipeline in short sections.
The helper modules under `src/vl_contradiction` hold the heavier logic so each cell stays readable.

## 1. Notebook Bootstrap

Use this section to resolve the project root, optionally install dependencies, and expose the package for imports.

In [3]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT_OVERRIDE = None  # Set this to a Drive path if auto-detection fails.

def _looks_like_project_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").exists()
        and (path / "src" / "vl_contradiction").exists()
        and (path / "configs" / "default.yaml").exists()
    )

def _candidate_roots() -> list[Path]:
    candidates = [Path.cwd().resolve()]
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
        candidates.extend([
            Path('/content/drive/MyDrive'),
            Path('/content/drive/MyDrive/comp646_multimodal_contradiction'),
        ])
    except ImportError:
        pass
    return candidates

def _find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if _looks_like_project_root(override):
            return override
        raise FileNotFoundError(f'PROJECT_ROOT_OVERRIDE does not point to a valid project root: {override}')

    for candidate in _candidate_roots():
        if _looks_like_project_root(candidate):
            return candidate
        for path in [candidate, *candidate.parents]:
            if _looks_like_project_root(path):
                return path
        for path in candidate.rglob('pyproject.toml'):
            root = path.parent
            if _looks_like_project_root(root):
                return root
    raise FileNotFoundError(
        'Could not locate the project root. Upload the full project folder to Drive and set '
        'PROJECT_ROOT_OVERRIDE to that folder if needed.'
    )

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], check=True)

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Could not locate the project root. Upload the full project folder to Drive and set PROJECT_ROOT_OVERRIDE to that folder if needed.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from vl_contradiction.audit import build_audit_sheet, summarize_audit
from vl_contradiction.benchmark import build_benchmark, sample_qwen_subset
from vl_contradiction.coco import ensure_coco_dataset, load_coco_caption_context
from vl_contradiction.config import load_config
from vl_contradiction.metrics import (
    bootstrap_macro_f1_ci,
    compute_classification_metrics,
    expected_calibration_error,
    fit_temperature,
)
from vl_contradiction.runtime import detect_runtime, ensure_directories, print_runtime_summary, set_global_seed


In [ ]:
from vl_contradiction.clip_baselines import (
    CLASS_ORDER,
    compute_similarity_scores,
    extract_joint_features,
    extract_token_features,
    fit_similarity_thresholds,
    load_clip_bundle,
    predict_with_thresholds,
)
from vl_contradiction.models import CrossAttentionFusionClassifier, LinearProbe
from vl_contradiction.plotting import (
    save_bar_chart,
    save_confusion_matrix,
    save_qualitative_panel,
    save_reliability_diagram,
    save_score_histogram,
    save_threshold_sweep,
    save_training_curves,
)
from vl_contradiction.qwen import load_qwen_bundle, run_qwen_inference
from vl_contradiction.training import FeatureDataset, TokenDataset, create_loader, evaluate_model, train_model


## 2. Configuration and Runtime

Load the shared config, resolve runtime paths, and create the artifact directories up front.

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "default.yaml")
runtime = detect_runtime(PROJECT_ROOT, config)
ensure_directories(
    [
        runtime.cache_root,
        runtime.dataset_root,
        runtime.benchmark_root,
        runtime.checkpoint_root,
        runtime.log_root,
        runtime.metrics_root,
        runtime.figure_root,
        runtime.qwen_root,
    ]
)
set_global_seed(config.runtime.seed)
print_runtime_summary(runtime)

In [ ]:
RUN_RAW_CLIP = True
RUN_LINEAR_PROBE = True
RUN_CROSS_ATTENTION = True
RUN_QWEN = False
CURRENT_STAGE = "prototype"

family_limit = getattr(config.data, f"{CURRENT_STAGE}_families")
print(f"Stage: {CURRENT_STAGE} | families: {family_limit}")

## 3. COCO Download and Validation

This section checks the Drive/local cache first and downloads COCO only when needed.

In [ ]:
coco_paths = ensure_coco_dataset(runtime.dataset_root, download=True)
print(coco_paths)

In [ ]:
coco_frame = load_coco_caption_context(coco_paths, splits=config.data.image_splits)
print(coco_frame.shape)
coco_frame.head(2)

## 4. Benchmark Construction

Build the benchmark from deterministic edits, then save the records and the family split manifest.

### Manual Audit Note

Fill in `label_valid` and `grammar_ok` in the generated audit CSV, then rerun the next cell.


In [ ]:
audit_sheet = build_audit_sheet(
    benchmark_frame,
    per_family=config.data.audit_samples_per_family,
    seed=config.runtime.seed,
)
audit_path = runtime.benchmark_root / f"audit_sheet_{CURRENT_STAGE}.csv"
audit_sheet.to_csv(audit_path, index=False)
print(f"Saved audit sheet to {audit_path}")
audit_sheet.head(10)

## 5. Split Views

Keep split-specific frames small and explicit so downstream sections stay easy to read.

In [ ]:
split_frames = {
    split: benchmark_frame.loc[benchmark_frame['split'] == split].reset_index(drop=True)
    for split in ['train', 'val', 'test']
}
for split, frame in split_frames.items():
    print(f"{split}: {frame.shape}")

## 6. Raw CLIP Baseline

Fit CLIP thresholds on validation only, then evaluate on test.

In [ ]:
clip_bundle = load_clip_bundle(config.model.clip_name, runtime.device)

In [ ]:
if RUN_RAW_CLIP:
    val_scores = compute_similarity_scores(split_frames['val'], clip_bundle, batch_size=config.training.batch_size)
    threshold_config, threshold_search = fit_similarity_thresholds(
        labels=val_scores['label'].tolist(),
        scores=val_scores['raw_score'].to_numpy(),
        grid_size=config.evaluation.threshold_grid_size,
    )
    test_scores = compute_similarity_scores(split_frames['test'], clip_bundle, batch_size=config.training.batch_size)
    test_predictions = predict_with_thresholds(
        test_scores['raw_score'].to_numpy(),
        tau_low=threshold_config['tau_low'],
        tau_high=threshold_config['tau_high'],
    )
    y_true = np.array([CLASS_ORDER.index(label) for label in test_scores['label']])
    clip_metrics = compute_classification_metrics(y_true, test_predictions)
    clip_ci = bootstrap_macro_f1_ci(y_true, test_predictions, samples=config.evaluation.bootstrap_samples, seed=config.runtime.seed)
    print("Raw CLIP metrics", clip_metrics)
    print("Raw CLIP macro-F1 95% CI", clip_ci)

In [ ]:
if RUN_RAW_CLIP:
    save_score_histogram(test_scores, runtime.figure_root / f"clip_scores_{CURRENT_STAGE}.pdf", "Raw CLIP Score Distribution")
    save_threshold_sweep(threshold_search, runtime.figure_root / f"clip_threshold_sweep_{CURRENT_STAGE}.pdf", "Validation Threshold Sweep")
    save_confusion_matrix(clip_metrics['confusion_matrix'], CLASS_ORDER, runtime.figure_root / f"clip_confusion_{CURRENT_STAGE}.pdf", "Raw CLIP Confusion Matrix")

    raw_clip_metrics_path = runtime.metrics_root / f"raw_clip_metrics_{CURRENT_STAGE}.json"
    raw_clip_metrics_path.write_text(json.dumps({**clip_metrics, 'macro_f1_ci': clip_ci, **threshold_config}, indent=2), encoding='utf-8')
    print(f"Saved raw CLIP metrics to {raw_clip_metrics_path}")

## 7. Frozen CLIP Linear Probe

Extract frozen features once, then train a small probe with per-epoch logging.

In [ ]:
if RUN_LINEAR_PROBE:
    feature_store = {}
    label_store = {}
    for split in ['train', 'val', 'test']:
        features, labels = extract_joint_features(split_frames[split], clip_bundle, batch_size=config.training.batch_size)
        feature_store[split] = features
        label_store[split] = labels
        print(f"Prepared {split} features: {tuple(features.shape)}")

In [ ]:
if RUN_LINEAR_PROBE:
    train_loader = create_loader(
        FeatureDataset(feature_store['train'], label_store['train']),
        batch_size=config.training.batch_size,
        shuffle=True,
        num_workers=config.training.num_workers,
    )
    val_loader = create_loader(
        FeatureDataset(feature_store['val'], label_store['val']),
        batch_size=config.training.batch_size,
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    linear_probe = LinearProbe(input_dim=feature_store['train'].shape[1])
    linear_result = train_model(
        model=linear_probe,
        train_loader=train_loader,
        val_loader=val_loader,
        device=runtime.device,
        epochs=config.training.epochs,
        learning_rate=config.training.learning_rate,
        weight_decay=config.training.weight_decay,
        log_dir=runtime.log_root / f"linear_probe_{CURRENT_STAGE}",
        checkpoint_path=runtime.checkpoint_root / f"linear_probe_{CURRENT_STAGE}.pt",
    )

In [ ]:
if RUN_LINEAR_PROBE:
    test_loader = create_loader(
        FeatureDataset(feature_store['test'], label_store['test']),
        batch_size=config.training.batch_size,
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    linear_metrics, _, _ = evaluate_model(linear_probe.to(runtime.device), test_loader, runtime.device)
    save_training_curves(linear_result.history, runtime.figure_root / f"linear_probe_training_{CURRENT_STAGE}.pdf", "Linear Probe Training")
    save_confusion_matrix(linear_metrics['confusion_matrix'], CLASS_ORDER, runtime.figure_root / f"linear_probe_confusion_{CURRENT_STAGE}.pdf", "Linear Probe Confusion Matrix")
    (runtime.metrics_root / f"linear_probe_metrics_{CURRENT_STAGE}.json").write_text(json.dumps(linear_metrics, indent=2), encoding='utf-8')
    print("Linear probe metrics", linear_metrics)

## 8. Cross-Attention Fusion

This section uses frozen CLIP token states and a lightweight cross-attention head.

In [ ]:
if RUN_CROSS_ATTENTION:
    token_store = {}
    token_labels = {}
    for split in ['train', 'val', 'test']:
        image_tokens, text_tokens, labels = extract_token_features(split_frames[split], clip_bundle, batch_size=max(1, config.training.batch_size // 2))
        token_store[split] = (image_tokens, text_tokens)
        token_labels[split] = labels
        print(f"Prepared {split} token tensors: image={tuple(image_tokens.shape)}, text={tuple(text_tokens.shape)}")

In [ ]:
if RUN_CROSS_ATTENTION:
    cross_attention = CrossAttentionFusionClassifier(
        input_dim=token_store['train'][0].shape[-1],
        hidden_dim=config.model.hidden_dim,
        num_heads=config.model.num_attention_heads,
        dropout=config.model.dropout,
    )
    cross_train_loader = create_loader(
        TokenDataset(token_store['train'][0], token_store['train'][1], token_labels['train']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=True,
        num_workers=config.training.num_workers,
    )
    cross_val_loader = create_loader(
        TokenDataset(token_store['val'][0], token_store['val'][1], token_labels['val']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    cross_result = train_model(
        model=cross_attention,
        train_loader=cross_train_loader,
        val_loader=cross_val_loader,
        device=runtime.device,
        epochs=config.training.epochs,
        learning_rate=config.training.learning_rate,
        weight_decay=config.training.weight_decay,
        log_dir=runtime.log_root / f"cross_attention_{CURRENT_STAGE}",
        checkpoint_path=runtime.checkpoint_root / f"cross_attention_{CURRENT_STAGE}.pt",
    )

In [ ]:
if RUN_CROSS_ATTENTION:
    cross_test_loader = create_loader(
        TokenDataset(token_store['test'][0], token_store['test'][1], token_labels['test']),
        batch_size=max(1, config.training.batch_size // 2),
        shuffle=False,
        num_workers=config.training.num_workers,
    )
    cross_metrics, cross_logits, cross_labels = evaluate_model(cross_attention.to(runtime.device), cross_test_loader, runtime.device)
    temperature = fit_temperature(cross_result.val_logits.to(runtime.device), cross_result.val_labels.to(runtime.device))
    calibrated_probs = torch.softmax(cross_logits / temperature, dim=1).cpu().numpy()
    calibration = expected_calibration_error(calibrated_probs, cross_labels.numpy())
    save_training_curves(cross_result.history, runtime.figure_root / f"cross_attention_training_{CURRENT_STAGE}.pdf", "Cross-Attention Training")
    save_confusion_matrix(cross_metrics['confusion_matrix'], CLASS_ORDER, runtime.figure_root / f"cross_attention_confusion_{CURRENT_STAGE}.pdf", "Cross-Attention Confusion Matrix")
    save_reliability_diagram(calibration.bin_centers, calibration.bin_accuracy, calibration.bin_confidence, runtime.figure_root / f"cross_attention_calibration_{CURRENT_STAGE}.pdf", "Cross-Attention Reliability")
    cross_metrics_path = runtime.metrics_root / f"cross_attention_metrics_{CURRENT_STAGE}.json"
    cross_metrics_path.write_text(json.dumps({**cross_metrics, 'temperature': temperature, 'ece': calibration.ece}, indent=2), encoding='utf-8')
    print("Cross-attention metrics", cross_metrics)
    print(f"Cross-attention calibration ECE: {calibration.ece:.4f}")

## 9. Qwen2.5-VL Zero-Shot Reference

This section runs only on the fixed subset and caches every raw output so reruns stay cheap.

In [ ]:
qwen_subset = sample_qwen_subset(benchmark_frame, subset_size=config.data.qwen_subset_size, seed=config.runtime.split_seed)
qwen_subset_path = runtime.benchmark_root / f"qwen_subset_{CURRENT_STAGE}.csv"
qwen_subset.to_csv(qwen_subset_path, index=False)
print(f"Saved Qwen subset to {qwen_subset_path}")
qwen_subset[['label', 'edit_family']].value_counts().head(12)

In [ ]:
if RUN_QWEN:
    qwen_bundle = load_qwen_bundle(config.model.qwen_name, use_4bit=config.model.use_qwen_4bit)
    qwen_outputs = run_qwen_inference(
        qwen_subset,
        qwen_bundle,
        output_dir=runtime.qwen_root / CURRENT_STAGE,
        max_new_tokens=config.model.max_qwen_tokens,
    )
    qwen_outputs['y_true'] = qwen_outputs['label'].map(CLASS_ORDER.index)
    qwen_outputs['y_pred'] = qwen_outputs['pred_label'].map(lambda label: CLASS_ORDER.index(label) if label in CLASS_ORDER else -1)
    qwen_eval = qwen_outputs.loc[qwen_outputs['y_pred'] >= 0].copy()
    qwen_metrics = compute_classification_metrics(qwen_eval['y_true'].to_numpy(), qwen_eval['y_pred'].to_numpy())
    (runtime.metrics_root / f"qwen_metrics_{CURRENT_STAGE}.json").write_text(json.dumps(qwen_metrics, indent=2), encoding='utf-8')
    print("Qwen metrics", qwen_metrics)

## 10. Qualitative and Per-Family Views

Use this section after model runs to export qualitative examples and summary plots.

In [ ]:
family_counts = benchmark_frame.groupby(['edit_family', 'label']).size().reset_index(name='count')
save_bar_chart(family_counts, 'edit_family', 'count', runtime.figure_root / f"benchmark_family_counts_{CURRENT_STAGE}.pdf", "Benchmark Distribution by Edit Family")
family_counts.head(12)

In [ ]:
qualitative_source = split_frames['test'].copy()
save_qualitative_panel(qualitative_source, runtime.figure_root / f"qualitative_panel_{CURRENT_STAGE}.pdf", "Prototype Qualitative Examples")
print(f"Saved qualitative panel to {runtime.figure_root / f'qualitative_panel_{CURRENT_STAGE}.pdf'}")

## 11. Summary of Generated Artifacts

This section gives a quick view of the benchmark, metrics, checkpoints, and figure outputs produced by the notebook.

In [ ]:
artifact_roots = [
    runtime.benchmark_root,
    runtime.metrics_root,
    runtime.figure_root,
    runtime.checkpoint_root,
    runtime.qwen_root,
]
for root in artifact_roots:
    print(f"\nArtifacts under {root}")
    for path in sorted(root.glob('*'))[:10]:
        print(f"  - {path.name}")